# Tarea Práctica: Gestión y Roles en la Integridad de Datos

## Contexto
Has ascendido a Data Engineer Senior en una plataforma de Streaming de Video. Tu empresa maneja millones de eventos (play, pause, stop) emitidos por distintas aplicaciones (Smart TVs, Móviles, Web). 

Tal y como estudiaste en la **UT3 - Capítulo 4**, la integridad no se logra solo con tecnología, sino con **gestión, contratos y responsabilidades claras**. Tu equipo (la capa de *Procesamiento y Catálogo*) no puede limpiar infinitamente la basura que envían los productores, y el equipo de BI (*Consumo*) no puede usar datos que no están certificados.

## Objetivo
Debes aplicar **Protocolos de Respuesta y Coherencia** sobre un lote de datos recién ingerido, decidiendo qué hacer con los problemas basándote en roles y responsabilidades.

Abordarás 3 retos típicos de gestión:
1.  **Responsabilidad del Productor (Garbage In):** Lidiar con eventos que violan el contrato mínimo indispensable para ser procesados.
2.  **Protocolo de SLA y Late Events:** Gestionar eventos que rompieron el acuerdo de nivel de servicio (llegaron demasiado tarde para el cierre contable diario).
3.  **Zonas de Datos (Raw a Refined):** Promocionar datos de la capa Bronce a Plata haciendo cumplir un contrato de tipos estricto y separando la corrupción a Cuarentena.

### ¡IMPORTANTE! ⚠️
En esta práctica se evaluará tu capacidad de razonar, aunque la IA te de ese codigo. Debes **justificar** cada acción en un comentario de texto argumentando en base a la teoría de Roles y Protocolos de la UT3. En resumen, intenta ser curioso para aprender. 

In [1]:
import pandas as pd

# --- CÓDIGO DE GENERACIÓN DE DATOS (NO MODIFICAR) ---
# Simulamos un Data Lake (Zona Raw/Bronce) donde acaba de volcarse la "bolsa" de eventos de ayer y hoy por la mañana.

data_lake_raw = {
    'event_id': ['EV-01', 'EV-02', None, 'EV-04', 'EV-05', 'EV-06', 'EV-07'], # EV-03 perdió su ID en la App Móvil
    'app_source': ['iOS', 'Android', 'Web', 'iOS', 'SmartTV', 'iOS', 'Android'],
    'event_type': ['play', 'pause', 'play', 'stop', 'play', 'play', 'play'],
    'timestamp_event': [
        '2026-03-01 22:30:00', # Ayer noche (OK)
        '2026-03-01 23:15:00', # Ayer noche (OK)
        '2026-03-01 23:45:00', # Ayer noche, pero sin event_id (Violación de contrato)
        '2026-03-02 01:00:00', # Madrugada de hoy (Evento normal de hoy)
        '2026-03-01 20:00:00', # Ayer noche, pero LLEGÓ TARDE a la capa de ingesta (Late Event)
        '2026-03-02 08:30:00', # Hoy por la mañana
        '2026-03-02 09:00:00'
    ],
    'ingestion_time': [ # Cuándo lo recibió el servidor
        '2026-03-01 22:30:05',
        '2026-03-01 23:15:10',
        '2026-03-01 23:45:05',
        '2026-03-02 01:00:05',
        '2026-03-02 08:45:00', # ¡FÍJATE! El evento del SmartTV de ayer se quedó sin Wi-Fi y llegó al servidor hoy a las 8:45 AM.
        '2026-03-02 08:30:05',
        '2026-03-02 09:00:02'
    ],
    'seconds_watched': [1200.0, 45.0, 300.0, 10.0, 5000.0, "ochenta", 60.0] # Un desarrollador de iOS mandó texto. ¡Corrupción silenciosa!
}

df_raw = pd.DataFrame(data_lake_raw)
# Convertimos strings a datetime para facilitar el filtrado
df_raw['timestamp_event'] = pd.to_datetime(df_raw['timestamp_event'])
df_raw['ingestion_time'] = pd.to_datetime(df_raw['ingestion_time'])

print("--- Data Lake (Capa Raw / Bronce) ---")
display(df_raw)

--- Data Lake (Capa Raw / Bronce) ---


,event_id,app_source,event_type,timestamp_event,ingestion_time,seconds_watched
0,EV-01,iOS,play,2026-03-01 22:30:00,2026-03-01 22:30:05,1200.0
1,EV-02,Android,pause,2026-03-01 23:15:00,2026-03-01 23:15:10,45.0
2,None,Web,play,2026-03-01 23:45:00,2026-03-01 23:45:05,300.0
3,EV-04,iOS,stop,2026-03-02 01:00:00,2026-03-02 01:00:05,10.0
4,EV-05,SmartTV,play,2026-03-01 20:00:00,2026-03-02 08:45:00,5000.0
5,EV-06,iOS,play,2026-03-02 08:30:00,2026-03-02 08:30:05,ochenta
6,EV-07,Android,play,2026-03-02 09:00:00,2026-03-02 09:00:02,60.0


---
### Ejercicio 1: Responsabilidad del Productor (El contrato mínimo)
*(Ref: UT3 puntos 4.1a y 4.1f)*

**Problema:** La teoría dice que la responsabilidad del Productor (las Apps) es generar IDs únicos y trazables. Si un evento no tiene `event_id`, no podemos saber si es un duplicado en el futuro. BI (Capa de consumo) no puede usar basura.

**Tarea:** 
1. Filtra y localiza el/los eventos que violan esta regla fundamental (aquellos con `event_id` nulo).
2. Sepáralos en un DataFrame `df_rechazados_origen` (para quejarnos con el equipo de front-end).
3. Mantén el resto en `df_raw_validado`.

In [ ]:
# La responsabilidad del Productor (las apps) es generar eventos con un ID
# unico. Es lo minimo. Sin event_id no podemos deduplicar en el futuro,
# ni trazar el evento, ni BI puede usarlo. Es "garbage in, garbage out".
#
# No relleno con un ID inventado porque seria como ponerle un DNI falso
# a alguien: tecnicamente "funciona" pero genera problemas peores.
# Lo rechazo y se reporta al equipo de front-end.
sin_id = df_raw["event_id"].isnull()
print(f"Eventos sin event_id: {sin_id.sum()}")
display(df_raw[sin_id])

df_rechazados_origen = df_raw[sin_id].copy()
df_raw_validado = df_raw[~sin_id].copy()

print(f"\nValidados: {len(df_raw_validado)}")
display(df_raw_validado)

---
### Ejercicio 2: Protocolo SLA y Late Events (Consistencia eventual)
*(Ref: UT3 punto 4.3 Ejemplo 2 y 4.2b)*

**Problema:** En la empresa hay un contrato de SLA: *"El cierre diario de métricas de ayer se consolida a las 08:00 AM de hoy. Cualquier evento de ayer que ingrese al almacén DESPUÉS de las 08:00 AM será procesado por separado para no cambiar los dashboards ya publicados"*.

**Tarea:**
Revisa el `df_raw_validado` (ya sin nulos). 
1. Encuentra el "Late Event": Un evento cuyo `timestamp_event` indique que ocurrió "ayer" (día 01), pero que su `ingestion_time` indique que nuestro servidor lo recibió "hoy" (día 02) DESPUÉS del umbral de las 08:00 AM.
2. Separa ese evento en un `df_late_events` para un reproceso contable secundario, y deja el resto listos en `df_para_silver`.

In [ ]:
# El SLA dice: metricas de ayer se cierran hoy a las 08:00 AM.
# Si un evento de ayer llega despues, es un Late Event.
# No lo metemos con los demas porque el dashboard de ayer ya se publico
# y si cambiamos las cifras, los jefes ven numeros distintos cada vez
# que miran y pierden la confianza en los datos.
#
# EV-05 ocurrio ayer (dia 01) a las 20:00 pero el SmartTV se quedo sin
# WiFi y llego al servidor hoy a las 08:45, despues del umbral.
umbral_sla = pd.Timestamp("2026-03-02 08:00:00")
dia_ayer = pd.Timestamp("2026-03-01").date()

mask_late = (
    (df_raw_validado["timestamp_event"].dt.date == dia_ayer) &
    (df_raw_validado["ingestion_time"] > umbral_sla)
)

df_late_events = df_raw_validado[mask_late].copy()
df_para_silver = df_raw_validado[~mask_late].copy()

print(f"Late Events: {len(df_late_events)}")
display(df_late_events)
print(f"\nListos para Silver: {len(df_para_silver)}")
display(df_para_silver)

---
### Ejercicio 3: Certificación y Prevención (Hacia la Capa Silver)
*(Ref: UT3 puntos 4.2c, 4.3 Ejemplo 3 y 4.2d)*

**Problema:** Toca pasar `df_para_silver` de la zona Raw a la zona Refined (Plata). El contrato dice que la columna `seconds_watched` debe ser estrictamente numérica (Float) para poder sumar los minutos de visualización. Sin embargo, un update de la app de iOS ha colado texto (`"ochenta"`).

**Tarea:**
1. Intenta forzar la conversión de toda la columna `seconds_watched` a numérico usando `pd.to_numeric()`. Usa el parámetro que convierte los fallos imprevistos en `NaN` (para no romper en seco todo el pipeline).
2. Una vez hecho, el sistema de Detección debe aislar (Cuarentena) ese registro dañado para que no pase la certificación hacia Gold. Filtra los nulos en `seconds_watched` y guárdalos en `df_cuarentena`.
3. El dataframe resultante se llamará `df_silver_certificado` y estará completamente pulido. Muéstralo por pantalla.

In [ ]:
# Para pasar de Raw a Silver, seconds_watched tiene que ser float si o si.
# Si no, no podemos sumar minutos de visualizacion.
# Un dev de iOS mando "ochenta" como texto en vez del numero 80.
#
# pd.to_numeric con errors="coerce" es la clave: en vez de petar el
# pipeline entero cuando encuentra texto, lo pone NaN y sigue.
# Asi no pierdo los demas datos validos por un solo registro malo.
df_para_silver["seconds_watched"] = pd.to_numeric(
    df_para_silver["seconds_watched"], errors="coerce"
)

# Los NaN van a cuarentena para reportar al equipo de iOS
mask_nulos = df_para_silver["seconds_watched"].isna()
df_cuarentena = df_para_silver[mask_nulos].copy()
df_silver_certificado = df_para_silver[~mask_nulos].copy()

print(f"Cuarentena: {len(df_cuarentena)}")
display(df_cuarentena)

print("\n" + "=" * 60)
print("DATASET SILVER CERTIFICADO")
print("=" * 60)
display(df_silver_certificado)
print(f"\nTotal minutos: {df_silver_certificado['seconds_watched'].sum() / 60:.2f} min")